In [1]:
from openai import OpenAI, OpenAIError
from dotenv import load_dotenv
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Literal
import os
import json

load_dotenv(Path.home() / ".env")

client = OpenAI()

MODEL = "gpt-6-luna"



In [67]:
interview_prompt = """
you are a professional AI job interviewer
you make the interview for the user's chosen job position
you take into account description of vacancy if given one 
you ask ONE question per message 
you wait for the answer before asking the next question
you adapt your next questions to previous answers 
ask questions relevant to the target role and the user's experience 
during the interview make sure to cover:
- educational background 
- previous work experience and responsibilities
ask exactly 7 questions in total and make sure to cover all the things i provided
evaluate the candidate's performance on a scale from 0 to 100
base the score on answer quality, relevance, clarity and completeness.
do not reveal the final score or feedback during the interview
"""

In [46]:
target_role = input('enter your target job position')
job_description = input('paste the job description(optional):')

In [42]:
interview_history = [
    {
        'role':'system',
        'content':f"{interview_prompt}\n {target_role}\n {job_description}"
    }
]

In [ ]:
def interview_chat(user_message):
    try:
        global answer_count 
        interview_history.append({
            'role':'user',
            'content':user_message
        })
        answer_count += 1
        if answer_count >= 7:
            return evaluate_interview()
        response = client.responses.create(
            model = MODEL,
            input = interview_history
        )
        question = response.output_text
        interview_history.append(
        {
        'role': 'assistant',
        'content':question
        }
    )
        return(question)
    except OpenAIError as error:
        print(error)
        return None

In [48]:
print(interview_chat('hello, i am ready for my interview'))

Great, let’s begin. Could you tell me about your educational background and how you’ve balanced your studies with playing hockey?


In [15]:
print(interview_chat('i have an experience in html, css and JS and I created small websites'))

Could you tell me about one of those websites—what it was for, which features you built, and what part was most challenging?


In [27]:
answer_count = 0

In [76]:
def start_interview():
    try:
        global answer_count, interview_history
        answer_count = 0 
        interview_history = [
            {
                'role':'system',
                'content':f"{interview_prompt}\n {target_role}\n {job_description}"
            }
        ]
        response = client.responses.create(
        model = MODEL,
        input = interview_history + [
            {'role': 'user', 'content':'start the interview. ask the first question'}
        ])
        question = response.output_text
        interview_history.append(
            {
                'role':'assistant', 
                'content':question
            }
    )
        return(question)
    except OpenAIError as error:
        print(error)
        return None    

In [77]:
start_interview()

'**Question 1 of 7:** Could you describe your educational background and how it prepared you for a career as an AI engineer?'

In [78]:
interview_chat('i am completing the ironhack course in AI engineering ')

'**Question 2 of 7:** What AI engineering topics or projects have you worked on during the Ironhack course, and what was your role in them?'

In [60]:
interview_chat('i built an amazon review evaluating ptoject from scratch')

'How did you build the Amazon review evaluation project, and what tools or AI techniques did you use?'

In [61]:
interview_chat('i used python with tf-idf and linearsvc for sentyment analysis, js for the web application and distilbart for summarization')

'What previous work experience do you have, and what were your main responsibilities?'

In [62]:
interview_chat('the biggest challenge was the imbalanced dataset and i solved the problem using different techniques that are convenient while working with imbalanced datasets')

'You mentioned the challenge of handling an imbalanced dataset. What techniques did you try, and what previous work experience—if any—have you had, including your main responsibilities?'

In [63]:
interview_chat('i have no previous experience yet, but i am young and full of motivation and energy guy who wants to work and improve')

'Which techniques did you use to address the class imbalance, and how did you evaluate whether they improved your model?'

In [64]:
interview_chat('i used a balanced linearsvc model to handle class imbalance. i evaluated it using macro f1 score and accuracy, which treats all classes equally. the model achieved about 93% of accuracy and 0.55 macro f1 score`')

'How would you improve your review-analysis project next—for example, its macro F1 score, deployment, or usefulness to people reading reviews?'

In [65]:
interview_chat('i would improve the accuracy of my model, make the app faster and provide more useful summaries for users')

InterviewFeedback(score=62, strengths=['You described a relevant end-to-end project combining sentiment analysis, summarization, and a web application.', 'You named specific tools and methods: Python, TF-IDF, LinearSVC, JavaScript, and DistilBART.', 'You recognized class imbalance as a modeling challenge and reported using a class-weighted LinearSVC.', 'You used macro F1 alongside accuracy, showing awareness that accuracy alone may not give a complete picture.', 'You communicated motivation to learn and improve.'], weaknesses=['Some answers were very brief and did not explain your decisions, implementation, or individual contribution in much depth.', 'When asked about work experience and responsibilities, you initially answered with a technical challenge instead, so the question was not addressed directly at first.', 'The reported 93% accuracy alongside a 0.55 macro F1 suggests performance may be uneven across classes and deserves further investigation. Accuracy does not treat all clas

In [66]:
class InterviewFeedback(BaseModel):
    score: int = Field(ge=0, le=100)
    strengths: list[str]
    weaknesses: list[str]
    recommendations: list[str]

In [74]:
def evaluate_interview():
    try:
        response = client.responses.parse(
            model=MODEL,
            input=interview_history + [
                {
                 "role": "user",
                    "content": (
                        "give feedback on the interview. "
                        "evaluate the candidate's performance on a scale from 0 to 100. "
                        "base the score on answer quality, relevance, clarity and completeness. "
                        "describe strengths, weaknesses and provide recommendations for improvement."
                        "evaluate the interview based on four criteria: relevance, clarity, completeness and role-specific knowledge, award 0 to 25 points for each criterion and sum them to calculate the final score out of 100"
                    )
                }
            ],
            text_format=InterviewFeedback
        )

        feedback = response.output_parsed
        return feedback
    except OpenAIError as error:
        print(error)
        return None

In [75]:
print(evaluate_interview())

score=63 strengths=['Relevance — 18/25: Most answers related to AI engineering, especially the project description and model evaluation.', 'Clarity — 16/25: You communicated the main points understandably and named relevant tools, including Python, TF-IDF, LinearSVC, JavaScript, and DistilBART.', 'Role-specific knowledge — 17/25: You showed practical familiarity with sentiment analysis, class imbalance, and evaluation metrics. Reporting both accuracy and macro F1 was useful.'] weaknesses=['Completeness — 12/25: Several responses were brief and lacked detail. You mentioned an Ironhack course but did not explain your broader educational background or what you learned.', 'Your answer about the dataset challenge did not address the question about previous work experience. You later clarified that you have no professional experience, but could have made that clear sooner.', 'Your description of addressing class imbalance was vague. You named a balanced LinearSVC but did not specify its conf

In [ ]:
print(evaluate_interview())

score=63 strengths=['Relevance — 18/25: Most answers related to AI engineering, especially the project description and model evaluation.', 'Clarity — 16/25: You communicated the main points understandably and named relevant tools, including Python, TF-IDF, LinearSVC, JavaScript, and DistilBART.', 'Role-specific knowledge — 17/25: You showed practical familiarity with sentiment analysis, class imbalance, and evaluation metrics. Reporting both accuracy and macro F1 was useful.'] weaknesses=['Completeness — 12/25: Several responses were brief and lacked detail. You mentioned an Ironhack course but did not explain your broader educational background or what you learned.', 'Your answer about the dataset challenge did not address the question about previous work experience. You later clarified that you have no professional experience, but could have made that clear sooner.', 'Your description of addressing class imbalance was vague. You named a balanced LinearSVC but did not specify its conf

In [90]:
cv_review_prompt = """
you are a professional CV reviewer
analyze CV for chosen profession
check the content: experience, educational background, skills etc.
evaluate clarity 
check the formatting: structure and design
check the strengths and weaknesses
give concrete recommendations about improving cv
check grammatical or spelling mistakes 
consider the specific job position if provided
do not invent the candidate's experience, education or skills
"""

In [97]:
class CVFeedback(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    errors: list[str]
    missing_skills: list[str]
    recommendations: list[str]

In [80]:
target_cv_role = input('enter your target role here: ')
cv_text = input('paste your cv text here(optional):')

In [81]:
%pip install pypdf


Note: you may need to restart the kernel to use updated packages.


In [82]:
from pypdf import PdfReader

In [86]:
def extract_cv_text(pdf_path):
    reader = PdfReader(pdf_path)
    cv_text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        cv_text += page_text
    return cv_text

In [87]:
print(extract_cv_text("pdf.pdf"))

 Alex Morgan 
 Junior AI Engineer 
 Skills:  Python, Pandas, Scikit-learn, OpenAI API,  HTML, CSS, JavaScript. 
 Education 
 AI Engineering Bootcamp, Ironhack — 2026 
 Projects 
 Customer Review Analyzer:  Built a sentiment analysis  model using Python, TF-IDF and 
 LinearSVC. 
 AI Travel Advisor:  Created an AI chatbot using OpenAI  API. 
 Experience 
 Freelance Web Developer — 2025 
 Built small websites using HTML, CSS and JavaScript. 


In [101]:
def review_cv(cv_text, target_cv_role):
    if not cv_text.strip():
        return None
    try:
        response_cv = client.responses.parse(
            model = MODEL, 
            input = [
                {
                    'role':'system',
                    'content': cv_review_prompt
                },
                {
                    'role':'user',
                    'content':f"{cv_text}\n {target_cv_role}"
                }
            ],
            text_format = CVFeedback
        )
        feedback_cv = response_cv.output_parsed
        return(feedback_cv)
    except OpenAIError as error:
            print(error)
            return None 


In [98]:
cv_text = extract_cv_text('resume.pdf')
target_cv_role = 'Junior AI Engineer'
print(review_cv(cv_text, target_cv_role))

strengths=['The CV is targeted toward a Junior AI Engineer role and names relevant tools, including Python, Pandas, scikit-learn, and the OpenAI API.', 'The projects show practical exposure to both traditional machine learning and AI application development.', 'Freelance web development adds evidence of hands-on programming experience with HTML, CSS, and JavaScript.'] weaknesses=['Project descriptions are brief and do not explain the problem, the candidate’s specific contributions, or the results.', 'The experience section does not provide much detail about freelance work, such as the number or type of websites built, client needs, or outcomes.', 'The education entry lists 2026 without saying whether the bootcamp is planned, in progress, or completed.', 'The skills list does not indicate proficiency levels or show how each skill was used.', 'The CV lacks supporting details such as a portfolio, GitHub profile, or links to the listed projects.'] errors=['“Junior AI Engineer” appears agai

In [102]:
print(review_cv("", "Junior AI Engineer"))

None


In [103]:
print(review_cv(
    'i know Python and JavaScript. i studied AI Engineering at Ironhack.',
    'Junior AI Engineer'
))

strengths=['Python and JavaScript are relevant programming languages for a Junior AI Engineer role.', 'AI Engineering studies at Ironhack provide relevant training.'] weaknesses=['The information is too brief to show the scope or depth of your training and technical skills.', 'No work experience, projects, achievements, or education dates are provided, so your practical readiness for the role is difficult to assess.', 'There is not enough content to evaluate CV structure, design, or overall clarity.'] errors=['No spelling or grammatical errors are apparent in the information provided.', 'A full CV was not included, so its formatting and writing cannot be checked.'] missing_skills=['Your information does not mention machine-learning or AI frameworks, tools, or methods. Include the ones you have actually used, if applicable.', 'Your information does not mention projects, data skills, model evaluation, or deployment experience. Add any relevant experience you can substantiate.'] recommend

In [104]:
job_fit_prompt = """
you need to compare CV and description of the job offer
find the skills of candidate that match the job offer requirements 
find out the necessary skills that user does not have for specific job opening
evaluate the Job Match Score with scale 0-100
give the recommendations of how to improve
do not invent invent candidate's skills or experience
base the Job Match Score based on how well the candidate meets the job requirements 
"""

In [105]:
class JobFitFeedback(BaseModel):
    match_score : int = Field(ge=0, le=100)
    matching_skills: list[str]
    missing_skills: list[str]
    strengths: list[str]
    recommendations: list[str]


In [109]:
def analyze_job_fit(cv_text, job_description):
    if not cv_text.strip() or not job_description.strip():
        return None
    try:
        response_job_fit = client.responses.parse(
            model = MODEL,
            input = [
                {
                    'role':'system',
                    'content':job_fit_prompt
                },
                {
                    'role':'user',
                    'content':f"{cv_text}\n {job_description}"
                }
            ],
            text_format = JobFitFeedback
        )
        feedback_job_fit = response_job_fit.output_parsed
        return feedback_job_fit
    except OpenAIError as error:
        print(error)
        return None


In [107]:
job_description = """
we are looking for a Junior Ai engineer with Python, machine learning, FastAPI, Git and API integration skills.
experience with AI projects and deployment is a plus.
"""

In [108]:
print(analyze_job_fit(cv_text, job_description))

match_score=65 matching_skills=['Python', 'Machine learning: built a sentiment analysis model using TF-IDF and LinearSVC, and lists Scikit-learn', 'API integration: used the OpenAI API to create a chatbot', 'AI project experience'] missing_skills=['FastAPI', 'Git', 'Deployment experience is not listed; this is a preferred qualification'] strengths=['Has hands-on experience building two AI-related projects.', 'The sentiment analysis project demonstrates practical use of machine-learning techniques.', 'The AI Travel Advisor shows experience integrating an external AI API.', "Python and Scikit-learn align with the role's core technical requirements."] recommendations=['Build a small FastAPI service that exposes an endpoint for one of the existing AI projects.', 'Use Git for the project, with clear commits and a README explaining setup and usage.', 'Deploy the FastAPI project to a cloud platform and document the deployment.', 'Update the CV with any verified FastAPI, Git, or deployment wor

In [111]:
print(analyze_job_fit('', job_description))

None


In [112]:
learning_roadmap_prompt = """
create a learning plan for 30 days 
take into account candidate's profession
use the missing skills from job-fit analysis 
split studies for 4 weeks
for each week you need to show topics and practical exercises
do not recommend to study skills that candidate already has
make the learning plan realistic for a beginner and include clear, achievable goals 
"""

In [113]:
class WeekPlan(BaseModel):
    week_number: int
    topics: list[str]
    exercises: list[str]
    goals: str 

class LearningRoadmap(BaseModel):
    weeks: list[WeekPlan]

In [118]:
def generate_learning_roadmap(target_role, missing_skills, matching_skills):
    if not target_role.strip() or not missing_skills:
        return None
    try:
        response_roadmap = client.responses.parse(
            model = MODEL, 
            input = [
                {
                    'role':'system',
                    'content': learning_roadmap_prompt
                },
                {
                    'role':'user',
                    'content': f"{target_role}\n {missing_skills}\n {matching_skills}"
                }
            ],
        text_format = LearningRoadmap
        )
        roadmap = response_roadmap.output_parsed
        return(roadmap)
    except OpenAIError as error:
        print (error)
        return None

In [116]:
job_fit_result = analyze_job_fit(cv_text, job_description)

In [117]:
print(generate_learning_roadmap('Junior AI engineer', job_fit_result.missing_skills, job_fit_result.matching_skills))

weeks=[WeekPlan(week_number=1, topics=['Git fundamentals: repositories, commits, branches, merging, and resolving simple conflicts', 'Using GitHub to host a repository and document a project'], exercises=['Create a GitHub repository for a small service, add a README, and make at least five meaningful commits.', 'Create a feature branch, make a change, merge it into the main branch, and resolve a simple merge conflict.'], goals='Use Git confidently for basic version control and publish a clearly documented project repository.'), WeekPlan(week_number=2, topics=['FastAPI fundamentals: routes, request and response models, path and query parameters', 'Input validation and interactive API documentation', 'Basic error handling'], exercises=['Build a small FastAPI service with health-check and text-submission endpoints.', "Add request and response models, validate inputs, and test the endpoints through FastAPI's interactive documentation."], goals='Create and run a basic FastAPI application wi

In [119]:
print(generate_learning_roadmap("", ["FastAPI"], ["Python"]))

None


In [120]:
print(generate_learning_roadmap("Junior AI Engineer", [], ["Python"]))

None


In [121]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python
